In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
import logging
logging.getLogger("gensim").setLevel(logging.WARNING)

BASE_DIR   = Path(r"D:\Shikha\bias_project")
GARG_LISTS = BASE_DIR / "word_lists" / "garg_2018"
WEAT_LISTS = BASE_DIR / "word_lists" / "caliskan_2017"
OUTPUT_DIR = BASE_DIR / "outputs" / "embeddings" / "word2vec"
CORPUS_DIR = BASE_DIR / "corpus"

def load_word_list(fp):
    with open(fp) as f:
        return [w.strip().lower() for w in f if w.strip()]

male_pairs    = load_word_list(GARG_LISTS / "male_pairs.txt")
female_pairs  = load_word_list(GARG_LISTS / "female_pairs.txt")
career_terms  = load_word_list(WEAT_LISTS / "career_terms_weat6.txt")
family_terms  = load_word_list(WEAT_LISTS / "family_terms_weat6.txt")

import re
def tokenize(text):
    if not isinstance(text, str) or not text.strip():
        return []
    sentences = re.split(r'[.!?]+|\n+', text)
    result = []
    for sent in sentences:
        toks = simple_preprocess(sent, deacc=True, min_len=2)
        if len(toks) >= 5:
            result.append(toks)
    return result

# Build period-split corpus (same as Word2Vec notebook)
cr_df = pd.read_csv(CORPUS_DIR / "congressional_record" / "2024-01" / "speeches.csv")
ca_df = pd.read_csv(CORPUS_DIR / "chronicling_america" / "1920-1922" / "articles_full_ocr.csv")
ca_df = ca_df.drop_duplicates('item_id')
ca_df = ca_df[ca_df['text_length'] >= 2000].reset_index(drop=True)
print(f"Chronicling America: {len(ca_df)} unique full-OCR pages (>=2000 chars)")
gd_df = pd.read_csv(CORPUS_DIR / "gdelt" / "2024-01" / "gdelt_articles_with_text.csv")
gd_df = gd_df[gd_df['text'].notna() & (gd_df['text'] != '')]
rd_posts    = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "posts.csv")
rd_comments = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "comments.csv")

state_period_corpus = {}
def add(state, period, text):
    sents = tokenize(text)
    if sents:
        state_period_corpus.setdefault((state, period), []).extend(sents)

for _, r in ca_df.iterrows():
    add(str(r['state_abbr']), 'hist_1920s', str(r.get('text','')))
for _, r in rd_posts.iterrows():
    add(str(r['state_abbr']), 'modern_2020', str(r.get('title',''))+' '+str(r.get('selftext','')))
for _, r in rd_comments.iterrows():
    add(str(r['state_abbr']), 'modern_2020', str(r.get('body','')))
for _, r in cr_df.iterrows():
    add(str(r['state']), 'modern_2024', str(r.get('text','')))
for _, r in gd_df.iterrows():
    add(str(r['state']), 'modern_2024', str(r.get('text','')))

print(f"State-period cells: {len(state_period_corpus)}")

Chronicling America: 300 unique full-OCR pages (>=2000 chars)
State-period cells: 111


In [2]:
EXCLUDE = {'GU','PR','VI','AS','MP'}
N_PERMUTATIONS = 1000
SEED = 42

def cosine_sim(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    return 0.0 if n1 == 0 or n2 == 0 else np.dot(v1, v2) / (n1 * n2)

def get_present_vecs(model, words):
    vecs = {}
    for w in words:
        try:
            vecs[w] = model.wv[w]
        except KeyError:
            pass
    return vecs

def weat_statistic(X_vecs, Y_vecs, A_vecs, B_vecs):
    """Test statistic s(X,Y,A,B) = sum_x s(x) - sum_y s(y), where
       s(w) = mean cos(w,A) - mean cos(w,B). (Caliskan unstandardized stat
       for permutation; significance is invariant to standardization.)"""
    A = list(A_vecs.values()); B = list(B_vecs.values())
    def s(wv):
        return np.mean([cosine_sim(wv, a) for a in A]) - np.mean([cosine_sim(wv, b) for b in B])
    return sum(s(v) for v in X_vecs.values()) - sum(s(v) for v in Y_vecs.values())

def permutation_pvalue(X_vecs, Y_vecs, A_vecs, B_vecs, n_perm=1000, rng=None):
    """One-sided permutation p: fraction of partitions with stat >= observed."""
    if rng is None:
        rng = np.random.default_rng(SEED)
    observed = weat_statistic(X_vecs, Y_vecs, A_vecs, B_vecs)
    pooled = list(X_vecs.values()) + list(Y_vecs.values())
    nX = len(X_vecs)
    count = 0
    for _ in range(n_perm):
        perm = rng.permutation(len(pooled))
        Xp = {i: pooled[perm[i]] for i in range(nX)}
        Yp = {i: pooled[perm[i]] for i in range(nX, len(pooled))}
        if weat_statistic(Xp, Yp, A_vecs, B_vecs) >= observed:
            count += 1
    return observed, (count + 1) / (n_perm + 1)  # +1 smoothing

# Run per cell (single seed=42 model; permutation handles significance)
results_perm = []
print(f"{'State':<7} {'Period':<13} {'stat':>9} {'perm p':>9} {'sig':>5}")
print("-" * 48)

for (state, period) in sorted(state_period_corpus.keys()):
    if state in EXCLUDE:
        continue
    sents = state_period_corpus[(state, period)]
    if len(sents) < 50:
        continue
    model = Word2Vec(sentences=sents, vector_size=100, window=5,
                     min_count=1, workers=1, epochs=20, seed=SEED)
    Xv = get_present_vecs(model, male_pairs)
    Yv = get_present_vecs(model, female_pairs)
    Av = get_present_vecs(model, career_terms)
    Bv = get_present_vecs(model, family_terms)
    if len(Xv) < 3 or len(Yv) < 3 or len(Av) < 2 or len(Bv) < 2:
        continue
    rng = np.random.default_rng(SEED)
    stat, p = permutation_pvalue(Xv, Yv, Av, Bv, N_PERMUTATIONS, rng)
    sig = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""
    print(f"{state:<7} {period:<13} {stat:>9.4f} {p:>9.4f} {sig:>5}")
    results_perm.append({
        'state': state, 'period': period,
        'weat_statistic': round(stat, 4),
        'perm_pvalue': round(p, 4),
        'significant_05': p < 0.05,
        'n_sentences': len(sents)
    })

print(f"\nCells tested: {len(results_perm)}")
print(f"Significant at p<0.05: {sum(r['significant_05'] for r in results_perm)}")

State   Period             stat    perm p   sig
------------------------------------------------
AK      modern_2020      0.0764    0.6214      
AK      modern_2024     -0.0056    0.9710      
AL      modern_2020      0.0147    0.8002      
AL      modern_2024      0.0587    0.3506      
AR      modern_2020     -0.0328    0.8192      
AR      modern_2024     -0.0112    0.7323      
AZ      modern_2024     -0.0092    0.5574      
CA      hist_1920s       0.0001    0.8621      
CA      modern_2020     -0.0083    0.3307      
CA      modern_2024      0.0706    0.0619     *
CO      modern_2024      0.0135    0.0170    **
CT      modern_2020      0.0248    0.2258      
CT      modern_2024     -0.0111    0.6743      
DE      modern_2020      0.0085    0.0050   ***
DE      modern_2024     -0.0056    0.0390    **
FL      modern_2020      0.0396    0.4016      
FL      modern_2024     -0.2076    0.4935      
GA      modern_2020     -0.0592    0.7473      
GA      modern_2024     -0.0965    0.60

In [3]:
perm_df = pd.DataFrame(results_perm).sort_values('perm_pvalue')
OUT_PERM = OUTPUT_DIR / "permutation_tests_word2vec.csv"
perm_df.to_csv(OUT_PERM, index=False)

print("=== Permutation Test Summary ===")
print(f"Cells tested            : {len(perm_df)}")
print(f"Significant p<0.01      : {(perm_df['perm_pvalue']<0.01).sum()}")
print(f"Significant p<0.05      : {(perm_df['perm_pvalue']<0.05).sum()}")
print(f"Significant p<0.10      : {(perm_df['perm_pvalue']<0.10).sum()}")
print(f"\nMost significant cells:")
print(perm_df.head(8)[['state','period','weat_statistic','perm_pvalue']].to_string(index=False))
print(f"\nSaved to: {OUT_PERM}")
print(f"\nNote: 1000 permutations per cell, Caliskan (2017) method, one-sided.")

=== Permutation Test Summary ===
Cells tested            : 82
Significant p<0.01      : 1
Significant p<0.05      : 4
Significant p<0.10      : 7

Most significant cells:
state      period  weat_statistic  perm_pvalue
   DE modern_2020          0.0085       0.0050
   CO modern_2024          0.0135       0.0170
   DE modern_2024         -0.0056       0.0390
   TX  hist_1920s          0.0965       0.0470
   CA modern_2024          0.0706       0.0619
   MN modern_2024         -0.0512       0.0669
   NV modern_2024          0.0022       0.0979
   LA modern_2024          0.0092       0.1399

Saved to: D:\Shikha\bias_project\outputs\embeddings\word2vec\permutation_tests_word2vec.csv

Note: 1000 permutations per cell, Caliskan (2017) method, one-sided.
